# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?

Solution: The architectural cut separates the controlling process from the physical process it controls. The Manipulation Executive (ME) is classified as a dul:Agent because it represents the program responsible for interpreting requests, maintaining internal states, making decisions and issuing action tokens. Although it controls the robot, it is not itself a physical object and does not directly produce physical movement. The Body-World System (BWS), in contrast, belongs to the physical domain. It consists of the robot's body, including its links, joints, actuators and sensors, together with the relevant environment. Its behaviour is governed by physical processes such as motion, contact, friction and gravity. The two subsystems interact through a closed feedback loop. The executive sends action tokens to the BWS, which physically realizes them. Sensors then generate observations that are returned to the executive, allowing it to update its internal state and decide what to do next. Therefore, the executive influences the physical world only through the BWS; it does not act directly on objects in the environment. This separation is important because issuing an action does not guarantee its intended physical outcome. The result depends on the interaction between the robot's body and the world.

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.

Solution: The possible courses of a robot–world manipulation system can be represented as a forward-branching tree. Starting from the initial state \(x_0\), the executive issues an action token, but the physical outcome is not always deterministic. Different world conditions can produce different successor states even when the same action is executed.
For example, when the PR2 performs a position-controlled pull on a fridge door or container seal, two outcomes are possible:
- If the seal releases, the door opens and the robot can continue with the manipulation task.
- If the seal remains stuck, the same pull may damage the handle or prevent the intended operation.
Each possible outcome creates a new branch from the current state. Further actions and physical interactions can create additional branches, producing a tree of possible trajectories. The branching is only forward because each transition depends on the current state, the selected action and the disturbances or choices of the world. Once a physical transition has occurred, the system cannot return to an earlier state in the trajectory simply by reasoning about it. Recovery is possible, but it creates a new continuation rather than reversing the previous history. Thus, the tree represents all physically possible courses from the initial conditions, rather than only the course the executive expects to occur.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.

Solution: The two runs should be classified according to the manipulation problem's requirements, including both the intended outcome and the restrictions on how that outcome is achieved.
- Run A — PROH (Prohibited): Although the drink reaches Eva, the container handle is torn off, violating the P4 prevention requirement. A forbidden event takes priority over successful goal achievement. Therefore, the run is prohibited.
- Run B — SOL (Solution): The drink is placed within Eva's reach while maintaining licensed contact and avoiding the stated violation. Assuming the remaining task and timing requirements are also satisfied, this run is a solution.
The classifier evaluates the complete trajectory because a successful final state does not necessarily mean that the task was performed correctly. A robot might reach the requested outcome after damaging an object, spilling liquid or violating another safety constraint along the way. This is the reason, the manipulation problem considers the events and conditions throughout the execution, not just the final configuration. In particular, forbidden events have priority over goal achievement. This prevents a robot from being considered successful simply because it eventually reaches the desired result despite causing damage during execution.

4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.

Solution: Description Logic represents concepts as classes and relationships as roles. In the RWMS ontology, the PR2's head camera is modelled as a sensor that is part of the robot body. The robot body is, in turn, a component of the Body-World System, which is a component of the Robot-World Manipulation System.

Using the transitivity of dul:isPartOf, the reasoner can infer the larger part-whole relationships from these existing relationships:
\[
\text{Head Camera}
\rightarrow \text{Robot Body}
\rightarrow \text{BWS}
\rightarrow \text{RWMS}
\]

Consequently, the head camera is inferred to be part of the RWMS without explicitly defining every possible relationship between the camera and the larger system. This demonstrates the advantage of an ontology with well-defined relationships: a reasoner such as HermiT can derive additional knowledge from the axioms instead of requiring every relationship to be manually specified. It also makes the model easier to maintain when new components or relationships are introduced.

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.

Solution: A suitable competency question is:

Can the manipulation problem itself be classified as a component of the Robot-World Manipulation System?

Adding this classification makes the ontology inconsistent. The RWMS axioms restrict its components to the Manipulation Executive or physical objects. However, the Manipulation Problem is a description that specifies the requirements and criteria used to assess the robot's behaviour; it is not a physical component of the system. This distinction is important because the manipulation problem defines the standard against which the system is evaluated, rather than participating directly in the physical interaction. Treating it as a physical component would violate the ontology's class restrictions and the disjointness constraints defined under DUL. Description Logic is useful for representing concepts, relationships, class restrictions and logical entailments. However, it does not perform general numerical calculations involving physical quantities. For example, evaluating the Coulomb grip condition
\[
2\mu F \geq m(g+a)
\]

requires numerical values for friction, grip force, mass, gravity and acceleration.

Such calculations must therefore be handled by the Semantic Digital Twin's causal model, which represents and predicts physical interactions. The ontology can describe the relevant objects and relationships, but the causal model determines whether a particular grip is physically sufficient to prevent slipping.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.

Solution: The distinction between logical strength and physical truth is essential when validating a robot agent.

An axiomatization such as \(K_0\), which does not include a friction law, has limited explanatory and predictive capabilities. A reasoner can still answer structural questions about the robot, its components and their relationships. But it cannot determine whether a particular grip force is sufficient to hold a carton, because the relevant physical relationship is missing from the model. Adding explicit dynamical laws in \(K_1/K_2\) makes the model more informative. It allows the system to represent additional causal relationships and derive predictions about physical behaviour under the assumptions encoded in those laws. This increases the range of questions that the model can address, but it does not automatically establish that those assumptions describe the real world correctly. Competency queries test the logical strength of an axiomatization, they establish what can be inferred from its definitions and rules. Physical experiments test its truth by comparing the model's predictions with actual observations. For example, the robot could perform controlled grip and lift experiments with cartons of different masses and surface friction, measuring whether and under which conditions slipping occurs. Therefore, a stronger axiomatization provides more expressive and testable predictions, but experimental evidence is still required to establish whether those predictions correspond to the physical reality.

## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 